In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np

BASE_PATH = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

folders = [
    "processed/model_datasets",
    "processed/leaky_datasets",
    "results",
    "model_artifacts"
]

for folder in folders:
    path = os.path.join(BASE_PATH, folder)
    print(f"\n{folder}:")

    if os.path.isdir(path):
        files = sorted(os.listdir(path))
        print("\n".join(files[:30]))
        print(f"Total files: {len(files)}")
    else:
        print("Folder not found")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

processed/model_datasets:
leakage_free_10.csv
leakage_free_20.csv
leakage_free_30.csv
leakage_free_40.csv
Total files: 4

processed/leaky_datasets:
assessment_features_full_course.csv
leaky_10.csv
leaky_20.csv
leaky_30.csv
leaky_40.csv
vle_features_full_course.csv
Total files: 6

results:
Figure_H2_Assessment_SHAP_Trajectory.png
Figure_H2_Individual_Feature_SHAP_Inflation.png
assessment_temporal_audit.csv
dataset_audit_summary.csv
final_H1_H2_evidence_table.csv
final_H2_adverse_class_robustness.csv
leakage_exposure_by_cutoff.csv
leakage_free_dataset_summary.csv
logistic_all_cutoffs_performance.csv
logistic_baseline_comparison_10.csv
logistic_baseline_comparison_20.csv
logistic_baseline_comparison_30.csv
logistic_baseline_comparison_40.csv
logistic_h1_inflation_summary.csv
logistic_h3_all_cutoffs.csv
logistic_h3_confusion_10.csv
logistic_h3_confusion_20.csv
l

In [2]:
import os
import pandas as pd

BASE = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")
LF_DIR = os.path.join(BASE, "processed/model_datasets")
LEAKY_DIR = os.path.join(BASE, "processed/leaky_datasets")

for cutoff in [10, 20, 30, 40]:
    lf = pd.read_csv(
        os.path.join(LF_DIR, f"leakage_free_{cutoff}.csv")
    )
    leaky = pd.read_csv(
        os.path.join(LEAKY_DIR, f"leaky_{cutoff}.csv")
    )

    print(f"\n{'='*50}")
    print(f"CUTOFF: {cutoff}%")
    print(f"{'='*50}")

    print("Leakage-free shape:", lf.shape)
    print("Leaky shape:", leaky.shape)

    print("\nLeakage-free columns:")
    print(lf.columns.tolist())

    print("\nLeaky columns:")
    print(leaky.columns.tolist())

    if "id_student" in lf.columns and "id_student" in leaky.columns:
        print("\nUnique students (LF):", lf["id_student"].nunique())
        print("Unique students (Leaky):", leaky["id_student"].nunique())

    if "final_result" in lf.columns and "final_result" in leaky.columns:
        print("\nLF outcome distribution:")
        print(lf["final_result"].value_counts().to_dict())

        print("Leaky outcome distribution:")
        print(leaky["final_result"].value_counts().to_dict())


CUTOFF: 10%
Leakage-free shape: (27652, 31)
Leaky shape: (27652, 31)

Leakage-free columns:
['code_module', 'code_presentation', 'id_student', 'cutoff_day', 'final_result', 'vle_clicks_10', 'vle_active_days_10', 'vle_unique_sites_10', 'precourse_clicks', 'precourse_active_days', 'precourse_unique_sites', 'content_clicks_10', 'forum_clicks_10', 'navigation_clicks_10', 'assessments_submitted_10', 'tma_submitted_10', 'cma_submitted_10', 'late_submissions_10', 'mean_submission_delay_10', 'scored_assessments_10', 'mean_score_due_10', 'has_score_due_10', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'date_registration']

Leaky columns:
['code_module', 'code_presentation', 'id_student', 'cutoff_day', 'final_result', 'precourse_clicks', 'precourse_active_days', 'precourse_unique_sites', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'date_regis

In [3]:
import os
import pandas as pd

BASE = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

KEYS = ["code_module", "code_presentation", "id_student"]

for cutoff in [10, 20, 30, 40]:
    lf = pd.read_csv(
        f"{BASE}/processed/model_datasets/leakage_free_{cutoff}.csv"
    )
    leaky = pd.read_csv(
        f"{BASE}/processed/leaky_datasets/leaky_{cutoff}.csv"
    )

    lf_keys = lf[KEYS + ["final_result"]].sort_values(KEYS).reset_index(drop=True)
    leaky_keys = leaky[KEYS + ["final_result"]].sort_values(KEYS).reset_index(drop=True)

    print(f"{cutoff}% cutoff:")
    print("Unique LF registrations:", not lf.duplicated(KEYS).any())
    print("Unique leaky registrations:", not leaky.duplicated(KEYS).any())
    print("Exact registration and outcome match:", lf_keys.equals(leaky_keys))
    print()

print("POSSIBLE ORIGINAL SPLIT FILES:")

for root, dirs, files in os.walk(BASE):
    for filename in files:
        if any(word in filename.lower() for word in [
            "split", "master_student", "partition"
        ]):
            print(os.path.join(root, filename))

10% cutoff:
Unique LF registrations: True
Unique leaky registrations: True
Exact registration and outcome match: True

20% cutoff:
Unique LF registrations: True
Unique leaky registrations: True
Exact registration and outcome match: True

30% cutoff:
Unique LF registrations: True
Unique leaky registrations: True
Exact registration and outcome match: True

40% cutoff:
Unique LF registrations: True
Unique leaky registrations: True
Exact registration and outcome match: True

POSSIBLE ORIGINAL SPLIT FILES:
/content/drive/MyDrive/AI_Research_Paper_1/processed/splits/master_student_split.csv


In [4]:
import os
import json
import pandas as pd

BASE = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

# Load the original persistent student-level split
split_path = os.path.join(
    BASE, "processed/splits/master_student_split.csv"
)

split_df = pd.read_csv(split_path)

print("=== ORIGINAL STUDENT SPLIT ===")
print("Shape:", split_df.shape)
print("Columns:", split_df.columns.tolist())
print("\nFirst five rows:")
print(split_df.head().to_string(index=False))

if "split" in split_df.columns:
    print("\nSplit distribution:")
    print(split_df["split"].value_counts().to_string())

    if "id_student" in split_df.columns:
        print("\nStudents assigned more than once:",
              split_df["id_student"].duplicated().sum())

# Inspect the saved model metadata
metadata_path = os.path.join(
    BASE, "model_artifacts/model_metadata.json"
)

print("\n=== SAVED MODEL METADATA ===")

if os.path.exists(metadata_path):
    with open(metadata_path, "r") as f:
        metadata = json.load(f)

    print("Metadata keys:", list(metadata.keys()))

    for key in [
        "primary_features",
        "categorical_features",
        "numeric_features",
        "target_mapping"
    ]:
        if key in metadata:
            print(f"\n{key}:")
            print(metadata[key])
else:
    print("Model metadata file not found.")

=== ORIGINAL STUDENT SPLIT ===
Shape: (24954, 2)
Columns: ['id_student', 'split']

First five rows:
 id_student split
       6516 train
       8462 train
      11391 train
      23629 train
      23698 train

Split distribution:
split
train    19963
test      4991

Students assigned more than once: 0

=== SAVED MODEL METADATA ===
Metadata keys: ['primary_features', 'numeric_features', 'categorical_features', 'target_mapping', 'class_order', 'cutoffs']

primary_features:
['code_module', 'code_presentation', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'date_registration', 'precourse_clicks', 'precourse_active_days', 'precourse_unique_sites', 'vle_clicks', 'vle_active_days', 'vle_unique_sites', 'content_clicks', 'forum_clicks', 'navigation_clicks', 'assessments_submitted', 'tma_submitted', 'cma_submitted', 'late_submissions', 'mean_submission_delay', 'scored_assessments', 'mean_score', 'has_score']

categorical_features:
['code_module', 'code_pr

In [5]:
import os
import json
import pandas as pd

BASE = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

with open(
    os.path.join(BASE, "model_artifacts/model_metadata.json"), "r"
) as f:
    metadata = json.load(f)

primary_features = metadata["primary_features"]
categorical_features = metadata["categorical_features"]
numeric_features = metadata["numeric_features"]

print("Total predictors:", len(primary_features))
print("Categorical predictors:", len(categorical_features))
print("Numerical predictors:", len(numeric_features))

assert len(primary_features) == 25
assert len(categorical_features) == 5
assert len(numeric_features) == 20
assert set(primary_features) == (
    set(categorical_features) | set(numeric_features)
)
assert not (
    set(categorical_features) & set(numeric_features)
)

print("\nDATASET VALIDATION")

for cutoff in [10, 20, 30, 40]:
    for condition, folder, filename in [
        ("Leakage-free", "model_datasets", f"leakage_free_{cutoff}.csv"),
        ("Leaky", "leaky_datasets", f"leaky_{cutoff}.csv")
    ]:
        path = os.path.join(BASE, "processed", folder, filename)

        data = pd.read_csv(path)

        # Canonicalize cutoff-specific VLE and assessment columns
        rename_map = {}

        for feature in primary_features:
            if feature not in data.columns:
                candidate = f"{feature}_{cutoff}"
                if candidate in data.columns:
                    rename_map[candidate] = feature

        data = data.rename(columns=rename_map)

        missing = [
            feature for feature in primary_features
            if feature not in data.columns
        ]

        print(
            f"{cutoff}% | {condition}: "
            f"{len(primary_features) - len(missing)}/25 predictors found"
        )

        if missing:
            print("Missing:", missing)

Total predictors: 25
Categorical predictors: 5
Numerical predictors: 20

DATASET VALIDATION
10% | Leakage-free: 23/25 predictors found
Missing: ['mean_score', 'has_score']
10% | Leaky: 11/25 predictors found
Missing: ['vle_clicks', 'vle_active_days', 'vle_unique_sites', 'content_clicks', 'forum_clicks', 'navigation_clicks', 'assessments_submitted', 'tma_submitted', 'cma_submitted', 'late_submissions', 'mean_submission_delay', 'scored_assessments', 'mean_score', 'has_score']
20% | Leakage-free: 23/25 predictors found
Missing: ['mean_score', 'has_score']
20% | Leaky: 11/25 predictors found
Missing: ['vle_clicks', 'vle_active_days', 'vle_unique_sites', 'content_clicks', 'forum_clicks', 'navigation_clicks', 'assessments_submitted', 'tma_submitted', 'cma_submitted', 'late_submissions', 'mean_submission_delay', 'scored_assessments', 'mean_score', 'has_score']
30% | Leakage-free: 23/25 predictors found
Missing: ['mean_score', 'has_score']
30% | Leaky: 11/25 predictors found
Missing: ['vle_cli

In [6]:
import pandas as pd
import os
import json

BASE = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

with open(f"{BASE}/model_artifacts/model_metadata.json") as f:
    metadata = json.load(f)

expected = metadata["primary_features"]

lf = pd.read_csv(
    f"{BASE}/processed/model_datasets/leakage_free_10.csv"
)

leaky = pd.read_csv(
    f"{BASE}/processed/leaky_datasets/leaky_10.csv"
)

print("=== EXPECTED 25 FEATURES ===")
for feature in expected:
    print(feature)

print("\n=== LEAKAGE-FREE COLUMNS ===")
for col in lf.columns:
    print(col)

print("\n=== LEAKY COLUMNS ===")
for col in leaky.columns:
    print(col)

print("\n=== MISSING FROM LF (EXACT NAMES) ===")
print(sorted(set(expected) - set(lf.columns)))

print("\n=== MISSING FROM LEAKY (EXACT NAMES) ===")
print(sorted(set(expected) - set(leaky.columns)))

=== EXPECTED 25 FEATURES ===
code_module
code_presentation
highest_education
imd_band
age_band
num_of_prev_attempts
studied_credits
date_registration
precourse_clicks
precourse_active_days
precourse_unique_sites
vle_clicks
vle_active_days
vle_unique_sites
content_clicks
forum_clicks
navigation_clicks
assessments_submitted
tma_submitted
cma_submitted
late_submissions
mean_submission_delay
scored_assessments
mean_score
has_score

=== LEAKAGE-FREE COLUMNS ===
code_module
code_presentation
id_student
cutoff_day
final_result
vle_clicks_10
vle_active_days_10
vle_unique_sites_10
precourse_clicks
precourse_active_days
precourse_unique_sites
content_clicks_10
forum_clicks_10
navigation_clicks_10
assessments_submitted_10
tma_submitted_10
cma_submitted_10
late_submissions_10
mean_submission_delay_10
scored_assessments_10
mean_score_due_10
has_score_due_10
gender
region
highest_education
imd_band
age_band
num_of_prev_attempts
studied_credits
disability
date_registration

=== LEAKY COLUMNS ===
code

In [7]:
import os
import pandas as pd

BASE = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

# Use metadata loaded in the previous step
features = metadata["primary_features"]

# Explicit mapping for assessment-score features
def canonicalize_dataset(df, cutoff, condition):
    df = df.copy()

    rename_map = {}

    for feature in features:
        if feature in df.columns:
            continue

        if condition == "LF":
            if feature in ["mean_score", "has_score"]:
                candidate = f"{feature}_due_{cutoff}"
            else:
                candidate = f"{feature}_{cutoff}"
        else:
            candidate = f"{feature}_full"

        if candidate in df.columns:
            rename_map[candidate] = feature

    df = df.rename(columns=rename_map)

    missing = [f for f in features if f not in df.columns]

    if missing:
        raise ValueError(
            f"{cutoff}% {condition}: Missing predictors: {missing}"
        )

    if df.columns.duplicated().any():
        raise ValueError("Duplicate column names detected")

    return df


# Validate all paired datasets
for cutoff in [10, 20, 30, 40]:

    lf_raw = pd.read_csv(
        f"{BASE}/processed/model_datasets/leakage_free_{cutoff}.csv"
    )

    leaky_raw = pd.read_csv(
        f"{BASE}/processed/leaky_datasets/leaky_{cutoff}.csv"
    )

    lf = canonicalize_dataset(lf_raw, cutoff, "LF")
    leaky = canonicalize_dataset(leaky_raw, cutoff, "Leaky")

    assert set(features).issubset(lf.columns)
    assert set(features).issubset(leaky.columns)

    print(
        f"{cutoff}% cutoff: "
        f"LF = {len(features)}/25, "
        f"Leaky = {len(features)}/25 "
        f"— PASSED"
    )

print("\nAll predictor mappings validated successfully!")

10% cutoff: LF = 25/25, Leaky = 25/25 — PASSED
20% cutoff: LF = 25/25, Leaky = 25/25 — PASSED
30% cutoff: LF = 25/25, Leaky = 25/25 — PASSED
40% cutoff: LF = 25/25, Leaky = 25/25 — PASSED

All predictor mappings validated successfully!


In [8]:
import pandas as pd
import os

BASE = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

split_df = pd.read_csv(
    f"{BASE}/processed/splits/master_student_split.csv"
)

CUT_OFFS = [10, 20, 30, 40]

prepared_data = {}

for cutoff in CUT_OFFS:
    prepared_data[cutoff] = {}

    for condition, folder, filename in [
        ("LF", "model_datasets", f"leakage_free_{cutoff}.csv"),
        ("Leaky", "leaky_datasets", f"leaky_{cutoff}.csv")
    ]:
        raw = pd.read_csv(
            f"{BASE}/processed/{folder}/{filename}"
        )

        data = canonicalize_dataset(raw, cutoff, condition)

        data = data.merge(
            split_df,
            on="id_student",
            how="left",
            validate="many_to_one"
        )

        assert data["split"].notna().all(), "Unassigned students found"

        train = data[data["split"] == "train"].copy()
        test = data[data["split"] == "test"].copy()

        assert set(train["id_student"]).isdisjoint(
            set(test["id_student"])
        ), "Student leakage detected"

        prepared_data[cutoff][condition] = {
            "train": train,
            "test": test
        }

        print(
            f"{cutoff}% {condition}: "
            f"Train={len(train):,}, "
            f"Test={len(test):,}"
        )

    # Verify both conditions use identical train/test registrations
    for partition in ["train", "test"]:
        lf = prepared_data[cutoff]["LF"][partition]
        leaky = prepared_data[cutoff]["Leaky"][partition]

        keys = ["code_module", "code_presentation",
                "id_student", "final_result"]

        lf_keys = lf[keys].sort_values(keys).reset_index(drop=True)
        leaky_keys = leaky[keys].sort_values(keys).reset_index(drop=True)

        assert lf_keys.equals(leaky_keys), (
            f"{cutoff}% {partition}: paired datasets do not match"
        )

    print(f"{cutoff}%: PAIRING VERIFIED\n")

print("All student-level partitions verified successfully!")

10% LF: Train=22,138, Test=5,514
10% Leaky: Train=22,138, Test=5,514
10%: PAIRING VERIFIED

20% LF: Train=21,274, Test=5,321
20% Leaky: Train=21,274, Test=5,321
20%: PAIRING VERIFIED

30% LF: Train=20,619, Test=5,164
30% Leaky: Train=20,619, Test=5,164
30%: PAIRING VERIFIED

40% LF: Train=20,091, Test=5,028
40% Leaky: Train=20,091, Test=5,028
40%: PAIRING VERIFIED

All student-level partitions verified successfully!


In [9]:

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from xgboost import XGBClassifier
import sklearn
import xgboost

print("Scikit-learn version:", sklearn.__version__)
print("XGBoost version:", xgboost.__version__)

# Model feature definitions from original metadata
features = metadata["primary_features"]
categorical = metadata["categorical_features"]
numeric = metadata["numeric_features"]
target_mapping = metadata["target_mapping"]

# Training-only preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            SimpleImputer(strategy="median"),
            numeric
        ),
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(
                    strategy="most_frequent"
                )),
                ("encoder", OneHotEncoder(
                    handle_unknown="ignore"
                ))
            ]),
            categorical
        )
    ]
)

# XGBoost classification model
model = XGBClassifier(
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=1,
    tree_method="hist"
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model)
])

# Grouped cross-validation
cv = StratifiedGroupKFold(
    n_splits=3,
    shuffle=True,
    random_state=42
)

print("\nPipeline created successfully!")
print("Predictors:", len(features))
print("Categorical:", len(categorical))
print("Numeric:", len(numeric))
print("CV folds:", cv.n_splits)

# Check all classes in the 10% training data
train_10 = prepared_data[10]["LF"]["train"]

print("\n10% training outcome distribution:")
print(train_10["final_result"].value_counts())

print("\nMapped outcomes:")
print(train_10["final_result"].map(target_mapping).value_counts())


Scikit-learn version: 1.6.1
XGBoost version: 3.4.1

Pipeline created successfully!
Predictors: 25
Categorical: 5
Numeric: 20
CV folds: 3

10% training outcome distribution:
final_result
Pass           9882
Fail           5570
Withdrawn      4244
Distinction    2442
Name: count, dtype: int64

Mapped outcomes:
final_result
1    9882
2    5570
3    4244
0    2442
Name: count, dtype: int64


In [10]:

from scipy.stats import randint, uniform
from sklearn.model_selection import RandomizedSearchCV

# Hyperparameter distributions
param_distributions = {
    "model__n_estimators": [200, 300, 500],
    "model__max_depth": [3, 5, 6],
    "model__learning_rate": [0.05, 0.08, 0.10],
    "model__min_child_weight": [1, 5, 7],
    "model__subsample": [0.7, 0.9, 1.0],
    "model__colsample_bytree": [0.7, 0.9, 1.0],
    "model__gamma": [0, 0.1, 0.3],
    "model__reg_alpha": [0, 0.01, 0.1],
    "model__reg_lambda": [1, 2, 10]
}

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=15,
    scoring="f1_macro",
    cv=cv,
    random_state=42,
    n_jobs=2,
    verbose=1,
    refit=True,
    error_score="raise"
)

X_train = train_10[features]
y_train = train_10["final_result"].map(target_mapping)
groups = train_10["id_student"]

assert y_train.notna().all()
assert X_train.shape[1] == 25
assert len(X_train) == len(y_train) == len(groups)

# Verify no student overlaps between CV training and validation
for fold, (tr_idx, val_idx) in enumerate(
    cv.split(X_train, y_train, groups=groups), start=1
):
    tr_students = set(groups.iloc[tr_idx])
    val_students = set(groups.iloc[val_idx])

    assert tr_students.isdisjoint(val_students)
    print(
        f"Fold {fold}: "
        f"Train={len(tr_idx):,}, "
        f"Validation={len(val_idx):,}, "
        f"Student overlap=0"
    )

print("\nRandomizedSearchCV configured successfully!")
print("Candidate configurations:", search.n_iter)
print("CV folds:", cv.n_splits)
print("Expected CV fits:", search.n_iter * cv.n_splits)


Fold 1: Train=14,790, Validation=7,348, Student overlap=0
Fold 2: Train=14,752, Validation=7,386, Student overlap=0
Fold 3: Train=14,734, Validation=7,404, Student overlap=0

RandomizedSearchCV configured successfully!
Candidate configurations: 15
CV folds: 3
Expected CV fits: 45


In [11]:

import os
import json
import pandas as pd
import joblib
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

# Save new results separately from the original experiments
OUTPUT_DIR = os.path.join(BASE, "results", "reviewer_sensitivity")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Use the original 10% leakage-free training data
train = prepared_data[10]["LF"]["train"]
test = prepared_data[10]["LF"]["test"]

X_train = train[features]
y_train = train["final_result"].map(target_mapping).astype(int)
groups = train["id_student"]

X_test = test[features]
y_test = test["final_result"].map(target_mapping).astype(int)

print("Starting independent XGBoost tuning: 10% leakage-free")
print("Training registrations:", len(X_train))
print("Testing registrations:", len(X_test))
print("CV fits:", search.n_iter * cv.n_splits)

# Fit only on training data using student-grouped CV
search.fit(X_train, y_train, groups=groups)

# Evaluate on untouched held-out test registrations
pred = search.best_estimator_.predict(X_test)

metrics = {
    "cutoff": 10,
    "condition": "Leakage-free",
    "accuracy": float(accuracy_score(y_test, pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
    "macro_f1": float(f1_score(y_test, pred, average="macro")),
    "best_cv_macro_f1": float(search.best_score_)
}

print("\nBEST HYPERPARAMETERS:")
print(search.best_params_)

print("\nHELD-OUT TEST RESULTS:")
print(metrics)

# Save tuning results and model
pd.DataFrame(search.cv_results_).to_csv(
    os.path.join(OUTPUT_DIR, "tuning_10pct_LF.csv"),
    index=False
)

pd.DataFrame([metrics]).to_csv(
    os.path.join(OUTPUT_DIR, "metrics_10pct_LF.csv"),
    index=False
)

joblib.dump(
    search.best_estimator_,
    os.path.join(OUTPUT_DIR, "xgboost_tuned_10pct_LF.joblib")
)

with open(
    os.path.join(OUTPUT_DIR, "best_params_10pct_LF.json"), "w"
) as f:
    json.dump(search.best_params_, f, indent=2)

print("\nExperiment completed and results saved!")


Starting independent XGBoost tuning: 10% leakage-free
Training registrations: 22138
Testing registrations: 5514
CV fits: 45
Fitting 3 folds for each of 15 candidates, totalling 45 fits

BEST HYPERPARAMETERS:
{'model__subsample': 0.7, 'model__reg_lambda': 2, 'model__reg_alpha': 0.01, 'model__n_estimators': 500, 'model__min_child_weight': 7, 'model__max_depth': 6, 'model__learning_rate': 0.05, 'model__gamma': 0.1, 'model__colsample_bytree': 1.0}

HELD-OUT TEST RESULTS:
{'cutoff': 10, 'condition': 'Leakage-free', 'accuracy': 0.5313746826260428, 'balanced_accuracy': 0.4071461765624364, 'macro_f1': 0.41976486727951645, 'best_cv_macro_f1': 0.40770597092018807}

Experiment completed and results saved!


In [12]:

from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import pandas as pd
import os
import json
import joblib

cutoff = 10

train = prepared_data[cutoff]["Leaky"]["train"]
test = prepared_data[cutoff]["Leaky"]["test"]

X_train = train[features]
y_train = train["final_result"].map(target_mapping).astype(int)
groups = train["id_student"]

X_test = test[features]
y_test = test["final_result"].map(target_mapping).astype(int)

# Create a fresh, independent search
leaky_search = clone(search)

print("Starting independent XGBoost tuning: 10% leaky")
print("Training registrations:", len(X_train))
print("Testing registrations:", len(X_test))

leaky_search.fit(X_train, y_train, groups=groups)

pred = leaky_search.best_estimator_.predict(X_test)

metrics = {
    "cutoff": cutoff,
    "condition": "Leaky",
    "accuracy": float(accuracy_score(y_test, pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
    "macro_f1": float(f1_score(y_test, pred, average="macro")),
    "best_cv_macro_f1": float(leaky_search.best_score_)
}

print("\nBEST HYPERPARAMETERS:")
print(leaky_search.best_params_)

print("\nHELD-OUT TEST RESULTS:")
for key, value in metrics.items():
    print(f"{key}: {value}")

# Save separately from original experiments
pd.DataFrame(leaky_search.cv_results_).to_csv(
    f"{OUTPUT_DIR}/tuning_10pct_Leaky.csv",
    index=False
)

pd.DataFrame([metrics]).to_csv(
    f"{OUTPUT_DIR}/metrics_10pct_Leaky.csv",
    index=False
)

joblib.dump(
    leaky_search.best_estimator_,
    f"{OUTPUT_DIR}/xgboost_tuned_10pct_Leaky.joblib"
)

with open(
    f"{OUTPUT_DIR}/best_params_10pct_Leaky.json", "w"
) as f:
    json.dump(leaky_search.best_params_, f, indent=2)

print("\n10% leaky experiment completed and saved!")


Starting independent XGBoost tuning: 10% leaky
Training registrations: 22138
Testing registrations: 5514
Fitting 3 folds for each of 15 candidates, totalling 45 fits

BEST HYPERPARAMETERS:
{'model__subsample': 0.7, 'model__reg_lambda': 1, 'model__reg_alpha': 0, 'model__n_estimators': 200, 'model__min_child_weight': 1, 'model__max_depth': 6, 'model__learning_rate': 0.05, 'model__gamma': 0.1, 'model__colsample_bytree': 1.0}

HELD-OUT TEST RESULTS:
cutoff: 10
condition: Leaky
accuracy: 0.7100108813928183
balanced_accuracy: 0.6300948968223705
macro_f1: 0.647185306439739
best_cv_macro_f1: 0.6346460623221718

10% leaky experiment completed and saved!


In [13]:

from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import pandas as pd
import json
import joblib
import os

all_results = []

for cutoff in [20, 30, 40]:
    for condition in ["LF", "Leaky"]:

        print(f"\n{'=' * 55}")
        print(f"TUNING: {cutoff}% — {condition}")
        print(f"{'=' * 55}")

        train = prepared_data[cutoff][condition]["train"]
        test = prepared_data[cutoff][condition]["test"]

        X_train = train[features]
        y_train = train["final_result"].map(target_mapping).astype(int)
        groups = train["id_student"]

        X_test = test[features]
        y_test = test["final_result"].map(target_mapping).astype(int)

        current_search = clone(search)
        current_search.fit(X_train, y_train, groups=groups)

        predictions = current_search.best_estimator_.predict(X_test)

        result = {
            "cutoff": cutoff,
            "condition": condition,
            "accuracy": accuracy_score(y_test, predictions),
            "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
            "macro_f1": f1_score(y_test, predictions, average="macro"),
            "best_cv_macro_f1": current_search.best_score_
        }

        all_results.append(result)

        suffix = f"{cutoff}pct_{condition}"

        pd.DataFrame(current_search.cv_results_).to_csv(
            f"{OUTPUT_DIR}/tuning_{suffix}.csv",
            index=False
        )

        pd.DataFrame([result]).to_csv(
            f"{OUTPUT_DIR}/metrics_{suffix}.csv",
            index=False
        )

        joblib.dump(
            current_search.best_estimator_,
            f"{OUTPUT_DIR}/xgboost_tuned_{suffix}.joblib"
        )

        with open(f"{OUTPUT_DIR}/best_params_{suffix}.json", "w") as f:
            json.dump(current_search.best_params_, f, indent=2)

        print(f"\nCompleted {cutoff}% {condition}")
        print(result)
        print("Results saved to Google Drive.")

summary = pd.DataFrame(all_results)
display(summary)

print("\nAll six additional tuning experiments completed!")



TUNING: 20% — LF
Fitting 3 folds for each of 15 candidates, totalling 45 fits

Completed 20% LF
{'cutoff': 20, 'condition': 'LF', 'accuracy': 0.5794023679759444, 'balanced_accuracy': np.float64(0.44277613794121223), 'macro_f1': 0.4586215137222674, 'best_cv_macro_f1': np.float64(0.43551091862642716)}
Results saved to Google Drive.

TUNING: 20% — Leaky
Fitting 3 folds for each of 15 candidates, totalling 45 fits

Completed 20% Leaky
{'cutoff': 20, 'condition': 'Leaky', 'accuracy': 0.710204848712648, 'balanced_accuracy': np.float64(0.6130724631299711), 'macro_f1': 0.6344834438888766, 'best_cv_macro_f1': np.float64(0.6264773633859626)}
Results saved to Google Drive.

TUNING: 30% — LF
Fitting 3 folds for each of 15 candidates, totalling 45 fits

Completed 30% LF
{'cutoff': 30, 'condition': 'LF', 'accuracy': 0.6097986057319907, 'balanced_accuracy': np.float64(0.46030738968504575), 'macro_f1': 0.4711978897106539, 'best_cv_macro_f1': np.float64(0.4605675758507865)}
Results saved to Google Dri

,cutoff,condition,accuracy,balanced_accuracy,macro_f1,best_cv_macro_f1
0,20,LF,0.579402,0.442776,0.458622,0.435511
1,20,Leaky,0.710205,0.613072,0.634483,0.626477
2,30,LF,0.609799,0.460307,0.471198,0.460568
3,30,Leaky,0.718435,0.601352,0.624622,0.627056
4,40,LF,0.642204,0.478792,0.490494,0.476955
5,40,Leaky,0.732697,0.586893,0.612436,0.614423



All six additional tuning experiments completed!


In [14]:

import os
import pandas as pd

rows = []

for cutoff in [10, 20, 30, 40]:
    for condition in ["LF", "Leaky"]:
        path = f"{OUTPUT_DIR}/metrics_{cutoff}pct_{condition}.csv"

        df = pd.read_csv(path)
        rows.append(df.iloc[0].to_dict())

combined = pd.DataFrame(rows)

combined["condition"] = combined["condition"].replace({
    "Leakage-free": "LF"
})

accuracy_table = combined.pivot(
    index="cutoff",
    columns="condition",
    values="accuracy"
)

accuracy_table["Inflation_pp"] = (
    accuracy_table["Leaky"] - accuracy_table["LF"]
) * 100

print("INDEPENDENT TUNING SENSITIVITY RESULTS")
print(accuracy_table.round(4).to_string())

combined.to_csv(
    f"{OUTPUT_DIR}/independent_tuning_all_cutoffs.csv",
    index=False
)

accuracy_table.to_csv(
    f"{OUTPUT_DIR}/independent_tuning_accuracy_comparison.csv"
)

print("\nConsolidated results saved successfully!")


INDEPENDENT TUNING SENSITIVITY RESULTS
condition      LF   Leaky  Inflation_pp
cutoff                                 
10         0.5314  0.7100       17.8636
20         0.5794  0.7102       13.0802
30         0.6098  0.7184       10.8637
40         0.6422  0.7327        9.0493

Consolidated results saved successfully!


In [15]:

# Define the feature families for leakage ablation

assessment_features = [
    "assessments_submitted",
    "tma_submitted",
    "cma_submitted",
    "late_submissions",
    "mean_submission_delay",
    "scored_assessments",
    "mean_score",
    "has_score"
]

vle_features = [
    "vle_clicks",
    "vle_active_days",
    "vle_unique_sites",
    "content_clicks",
    "forum_clicks",
    "navigation_clicks"
]

precourse_features = [
    "precourse_clicks",
    "precourse_active_days",
    "precourse_unique_sites"
]

static_features = [
    f for f in features
    if f not in assessment_features
    and f not in vle_features
    and f not in precourse_features
]

print("Assessment features:", len(assessment_features))
print("In-course VLE features:", len(vle_features))
print("Pre-course VLE features:", len(precourse_features))
print("Static/registration features:", len(static_features))

all_grouped = (
    assessment_features
    + vle_features
    + precourse_features
    + static_features
)

assert len(all_grouped) == 25
assert len(set(all_grouped)) == 25
assert set(all_grouped) == set(features)

print("\nAll 25 features assigned to exactly one family.")
print("\nStatic/registration features:")
print(static_features)


Assessment features: 8
In-course VLE features: 6
Pre-course VLE features: 3
Static/registration features: 8

All 25 features assigned to exactly one family.

Static/registration features:
['code_module', 'code_presentation', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'date_registration']


In [16]:

import pandas as pd
from pandas.testing import assert_frame_equal

KEYS = ["code_module", "code_presentation", "id_student"]

ablation_data = {}

for cutoff in [10, 20, 30, 40]:
    ablation_data[cutoff] = {}

    lf = canonicalize_dataset(
        pd.read_csv(
            f"{BASE}/processed/model_datasets/leakage_free_{cutoff}.csv"
        ),
        cutoff,
        "LF"
    )

    leaky = canonicalize_dataset(
        pd.read_csv(
            f"{BASE}/processed/leaky_datasets/leaky_{cutoff}.csv"
        ),
        cutoff,
        "Leaky"
    )

    # Align student–module registrations
    lf = lf.sort_values(KEYS).reset_index(drop=True)
    leaky = leaky.sort_values(KEYS).reset_index(drop=True)

    assert_frame_equal(
        lf[KEYS + ["final_result"]],
        leaky[KEYS + ["final_result"]]
    )

    # Four ablation conditions
    conditions = {
        "Leakage_Free": lf.copy(),
        "Assessment_Only": lf.copy(),
        "VLE_Only": lf.copy(),
        "Combined_Leaky": lf.copy()
    }

    # Substitute only assessment features
    conditions["Assessment_Only"][assessment_features] = (
        leaky[assessment_features].to_numpy()
    )

    # Substitute only in-course VLE features
    conditions["VLE_Only"][vle_features] = (
        leaky[vle_features].to_numpy()
    )

    # Substitute both feature families
    conditions["Combined_Leaky"][
        assessment_features + vle_features
    ] = leaky[
        assessment_features + vle_features
    ].to_numpy()

    # Verify intended substitutions and unchanged features
    for name, dataset in conditions.items():

        changed = {
            "Leakage_Free": [],
            "Assessment_Only": assessment_features,
            "VLE_Only": vle_features,
            "Combined_Leaky": assessment_features + vle_features
        }[name]

        unchanged = [f for f in features if f not in changed]

        assert_frame_equal(
            dataset[unchanged],
            lf[unchanged],
            check_dtype=False
        )

        if changed:
            assert_frame_equal(
                dataset[changed],
                leaky[changed],
                check_dtype=False
            )

        assert_frame_equal(
            dataset[KEYS + ["final_result"]],
            lf[KEYS + ["final_result"]]
        )

        ablation_data[cutoff][name] = dataset

    # Confirm combined condition equals original leaky predictors
    assert_frame_equal(
        conditions["Combined_Leaky"][features],
        leaky[features],
        check_dtype=False
    )

    print(f"{cutoff}%: All four ablation conditions verified.")

print("\nFeature-family ablation datasets constructed successfully!")


10%: All four ablation conditions verified.
20%: All four ablation conditions verified.
30%: All four ablation conditions verified.
40%: All four ablation conditions verified.

Feature-family ablation datasets constructed successfully!


In [17]:

from pandas.testing import assert_frame_equal

ablation_splits = {}

KEYS = ["code_module", "code_presentation",
        "id_student", "final_result"]

for cutoff in [10, 20, 30, 40]:
    ablation_splits[cutoff] = {}

    for condition, df in ablation_data[cutoff].items():

        data = df.merge(
            split_df,
            on="id_student",
            how="left",
            validate="many_to_one"
        )

        assert data["split"].notna().all()

        train = data[data["split"] == "train"].copy()
        test = data[data["split"] == "test"].copy()

        assert set(train["id_student"]).isdisjoint(
            set(test["id_student"])
        )

        ablation_splits[cutoff][condition] = {
            "train": train,
            "test": test
        }

    # Check every condition against the leakage-free baseline
    for partition in ["train", "test"]:
        baseline = ablation_splits[cutoff]["Leakage_Free"][partition]
        baseline_keys = baseline[KEYS].sort_values(KEYS).reset_index(drop=True)

        for condition in [
            "Assessment_Only", "VLE_Only", "Combined_Leaky"
        ]:
            comparison = ablation_splits[cutoff][condition][partition]
            comparison_keys = (
                comparison[KEYS].sort_values(KEYS).reset_index(drop=True)
            )

            assert_frame_equal(baseline_keys, comparison_keys)

    print(f"{cutoff}%: All four conditions use identical train/test registrations.")

print("\nAll ablation partitions verified successfully!")


10%: All four conditions use identical train/test registrations.
20%: All four conditions use identical train/test registrations.
30%: All four conditions use identical train/test registrations.
40%: All four conditions use identical train/test registrations.

All ablation partitions verified successfully!


In [18]:

from pandas.testing import assert_frame_equal

ablation_splits = {}

KEYS = ["code_module", "code_presentation",
        "id_student", "final_result"]

for cutoff in [10, 20, 30, 40]:
    ablation_splits[cutoff] = {}

    for condition, df in ablation_data[cutoff].items():

        data = df.merge(
            split_df,
            on="id_student",
            how="left",
            validate="many_to_one"
        )

        assert data["split"].notna().all()

        train = data[data["split"] == "train"].copy()
        test = data[data["split"] == "test"].copy()

        assert set(train["id_student"]).isdisjoint(
            set(test["id_student"])
        )

        ablation_splits[cutoff][condition] = {
            "train": train,
            "test": test
        }

    # Check every condition against the leakage-free baseline
    for partition in ["train", "test"]:
        baseline = ablation_splits[cutoff]["Leakage_Free"][partition]
        baseline_keys = baseline[KEYS].sort_values(KEYS).reset_index(drop=True)

        for condition in [
            "Assessment_Only", "VLE_Only", "Combined_Leaky"
        ]:
            comparison = ablation_splits[cutoff][condition][partition]
            comparison_keys = (
                comparison[KEYS].sort_values(KEYS).reset_index(drop=True)
            )

            assert_frame_equal(baseline_keys, comparison_keys)

    print(f"{cutoff}%: All four conditions use identical train/test registrations.")

print("\nAll ablation partitions verified successfully!")


10%: All four conditions use identical train/test registrations.
20%: All four conditions use identical train/test registrations.
30%: All four conditions use identical train/test registrations.
40%: All four conditions use identical train/test registrations.

All ablation partitions verified successfully!


In [19]:

import json
import os
import joblib

BASE = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")
ARTIFACTS = os.path.join(BASE, "model_artifacts")

print("SAVED MODEL FILES:")
for filename in sorted(os.listdir(ARTIFACTS)):
    if filename.endswith((".joblib", ".json")):
        print(filename)

print("\nORIGINAL XGBOOST PARAMETERS TO VERIFY:")

original_params = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.08,
    "min_child_weight": 7,
    "subsample": 0.7,
    "colsample_bytree": 1.0,
    "gamma": 0.1,
    "reg_alpha": 0.01,
    "reg_lambda": 10
}

for key, value in original_params.items():
    print(f"{key}: {value}")

print("\nCheck the saved original XGBoost model before training.")


SAVED MODEL FILES:
logistic_leaky_10.joblib
logistic_leaky_20.joblib
logistic_leaky_30.joblib
logistic_leaky_40.joblib
logistic_lf_10.joblib
logistic_lf_20.joblib
logistic_lf_30.joblib
logistic_lf_40.joblib
model_metadata.json
xgb_leaky_10.joblib
xgb_leaky_20.joblib
xgb_leaky_30.joblib
xgb_leaky_40.joblib
xgb_lf_10.joblib
xgb_lf_20.joblib
xgb_lf_30.joblib
xgb_lf_40.joblib

ORIGINAL XGBOOST PARAMETERS TO VERIFY:
n_estimators: 200
max_depth: 6
learning_rate: 0.08
min_child_weight: 7
subsample: 0.7
colsample_bytree: 1.0
gamma: 0.1
reg_alpha: 0.01
reg_lambda: 10

Check the saved original XGBoost model before training.


In [20]:

import os
import joblib
import math

BASE = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

original_params = {
    "n_estimators": 200,
    "max_depth": 6,
    "learning_rate": 0.08,
    "min_child_weight": 7,
    "subsample": 0.7,
    "colsample_bytree": 1.0,
    "gamma": 0.1,
    "reg_alpha": 0.01,
    "reg_lambda": 10
}

all_match = True

for cutoff in [10, 20, 30, 40]:
    for condition in ["lf", "leaky"]:

        path = (
            f"{BASE}/model_artifacts/"
            f"xgb_{condition}_{cutoff}.joblib"
        )

        saved = joblib.load(path)

        # Original saved object may be a pipeline
        model = (
            saved.named_steps["model"]
            if hasattr(saved, "named_steps")
            else saved
        )

        actual = model.get_params()

        differences = {
            key: (expected, actual.get(key))
            for key, expected in original_params.items()
            if not (
                actual.get(key) is not None
                and math.isclose(
                    float(actual[key]),
                    float(expected),
                    rel_tol=1e-9,
                    abs_tol=1e-9
                )
            )
        }

        passed = len(differences) == 0
        all_match = all_match and passed

        print(
            f"{cutoff}% {condition}: "
            f"{'MATCH' if passed else 'DIFFERENT'}"
        )

        if differences:
            print("Differences:", differences)

print("\nAll models match:", all_match)


/usr/lib/python3.13/pickle.py:1754: UserWarning: [11:06:48] WARNING: /__w/xgboost/xgboost/src/collective/../data/../common/error_msg.h:83: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  setstate(state)


10% lf: DIFFERENT
Differences: {'n_estimators': (200, 500), 'max_depth': (6, 3), 'learning_rate': (0.08, 0.05), 'min_child_weight': (7, 3), 'reg_alpha': (0.01, 0.1), 'reg_lambda': (10, 2)}
10% leaky: DIFFERENT
Differences: {'n_estimators': (200, 500), 'max_depth': (6, 3), 'learning_rate': (0.08, 0.05), 'min_child_weight': (7, 3), 'reg_alpha': (0.01, 0.1), 'reg_lambda': (10, 2)}
20% lf: DIFFERENT
Differences: {'n_estimators': (200, 500), 'max_depth': (6, 3), 'learning_rate': (0.08, 0.05), 'min_child_weight': (7, 3), 'reg_alpha': (0.01, 0.1), 'reg_lambda': (10, 2)}
20% leaky: DIFFERENT
Differences: {'n_estimators': (200, 500), 'max_depth': (6, 3), 'learning_rate': (0.08, 0.05), 'min_child_weight': (7, 3), 'reg_alpha': (0.01, 0.1), 'reg_lambda': (10, 2)}
30% lf: DIFFERENT
Differences: {'n_estimators': (200, 500), 'max_depth': (6, 3), 'learning_rate': (0.08, 0.05), 'min_child_weight': (7, 3), 'reg_alpha': (0.01, 0.1), 'reg_lambda': (10, 2)}
30% leaky: DIFFERENT
Differences: {'n_estimators'

In [21]:

import joblib

path = f"{BASE}/model_artifacts/xgb_lf_10.joblib"
saved = joblib.load(path)

model = (
    saved.named_steps["model"]
    if hasattr(saved, "named_steps")
    else saved
)

params = model.get_params()

keys = [
    "n_estimators", "max_depth", "learning_rate",
    "min_child_weight", "subsample",
    "colsample_bytree", "gamma",
    "reg_alpha", "reg_lambda",
    "objective", "eval_metric", "random_state"
]

print("ACTUAL ORIGINAL XGBOOST SETTINGS")
for key in keys:
    print(f"{key}: {params.get(key)}")

print("\nSaved object type:", type(saved).__name__)
print("Model fitted:", hasattr(model, "classes_"))


ACTUAL ORIGINAL XGBOOST SETTINGS
n_estimators: 500
max_depth: 3
learning_rate: 0.05
min_child_weight: 3
subsample: 0.7
colsample_bytree: 1.0
gamma: 0.1
reg_alpha: 0.1
reg_lambda: 2
objective: multi:softprob
eval_metric: mlogloss
random_state: 42

Saved object type: Pipeline
Model fitted: True


In [22]:

import joblib

path = f"{BASE}/model_artifacts/xgb_lf_10.joblib"
original_pipeline = joblib.load(path)

print("ORIGINAL PIPELINE STEPS:")
print(list(original_pipeline.named_steps.keys()))

print("\nPREPROCESSING STRUCTURE:")
for name, step in original_pipeline.named_steps.items():
    print(f"\n{name}:")
    if name != "model":
        print(step)

print("\nXGBOOST MODEL:")
print(type(original_pipeline.named_steps["model"]).__name__)


ORIGINAL PIPELINE STEPS:
['preprocessor', 'model']

PREPROCESSING STRUCTURE:

preprocessor:
ColumnTransformer(transformers=[('numeric',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median'))]),
                                 ['num_of_prev_attempts', 'studied_credits',
                                  'date_registration', 'precourse_clicks',
                                  'precourse_active_days',
                                  'precourse_unique_sites', 'vle_clicks',
                                  'vle_active_days', 'vle_unique_sites',
                                  'content_clicks', 'forum_clicks',
                                  'navigation_clicks', 'assessments_submitted',
                                  'tma_submitted', 'cma_submitted',
                                  'late_submissions', 'mean_submission_delay',
                                  'scored_assessments', 'mean_s

In [23]:

import os
import joblib
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

# Use original fitted model as a configuration template
original_pipeline = joblib.load(
    f"{BASE}/model_artifacts/xgb_lf_10.joblib"
)

# Clone removes fitted state but preserves model settings
ablation_model = clone(original_pipeline)

cutoff = 10
condition = "Assessment_Only"

train = ablation_splits[cutoff][condition]["train"]
test = ablation_splits[cutoff][condition]["test"]

X_train = train[features]
y_train = train["final_result"].map(target_mapping).astype(int)

X_test = test[features]
y_test = test["final_result"].map(target_mapping).astype(int)

assert X_train.shape[1] == 25
assert X_test.shape[1] == 25

assert set(train["id_student"]).isdisjoint(
    set(test["id_student"])
)

print("Training 10% assessment-only leakage model")
print("Training registrations:", len(train))
print("Testing registrations:", len(test))

# Fit preprocessing and classifier on training data only
ablation_model.fit(X_train, y_train)

predictions = ablation_model.predict(X_test)

result = {
    "cutoff": cutoff,
    "condition": condition,
    "accuracy": accuracy_score(y_test, predictions),
    "balanced_accuracy": balanced_accuracy_score(y_test, predictions),
    "macro_f1": f1_score(y_test, predictions, average="macro")
}

print("\nABLATION RESULTS:")
for key, value in result.items():
    print(f"{key}: {value}")

# Save independently from original experiments
ABLATION_DIR = os.path.join(
    BASE, "results", "reviewer_sensitivity", "ablation"
)
os.makedirs(ABLATION_DIR, exist_ok=True)

pd.DataFrame([result]).to_csv(
    f"{ABLATION_DIR}/metrics_10pct_Assessment_Only.csv",
    index=False
)

joblib.dump(
    ablation_model,
    f"{ABLATION_DIR}/model_10pct_Assessment_Only.joblib"
)

print("\nAssessment-only experiment completed and saved!")


Training 10% assessment-only leakage model
Training registrations: 22138
Testing registrations: 5514

ABLATION RESULTS:
cutoff: 10
condition: Assessment_Only
accuracy: 0.6987667754805948
balanced_accuracy: 0.6193759643544741
macro_f1: 0.6355439033465767

Assessment-only experiment completed and saved!


In [24]:

from sklearn.base import clone
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score
)
import pandas as pd
import joblib

cutoff = 10
condition = "VLE_Only"

train = ablation_splits[cutoff][condition]["train"]
test = ablation_splits[cutoff][condition]["test"]

X_train = train[features]
y_train = train["final_result"].map(target_mapping).astype(int)

X_test = test[features]
y_test = test["final_result"].map(target_mapping).astype(int)

# Fresh model, same original frozen hyperparameters
vle_model = clone(original_pipeline)

assert set(train["id_student"]).isdisjoint(
    set(test["id_student"])
)

print("Training 10% VLE-only leakage model")

vle_model.fit(X_train, y_train)

pred = vle_model.predict(X_test)

result = {
    "cutoff": cutoff,
    "condition": condition,
    "accuracy": float(accuracy_score(y_test, pred)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
    "macro_f1": float(f1_score(y_test, pred, average="macro"))
}

print("\nVLE-ONLY ABLATION RESULTS:")
for key, value in result.items():
    print(f"{key}: {value}")

pd.DataFrame([result]).to_csv(
    f"{ABLATION_DIR}/metrics_10pct_VLE_Only.csv",
    index=False
)

joblib.dump(
    vle_model,
    f"{ABLATION_DIR}/model_10pct_VLE_Only.joblib"
)

print("\n10% VLE-only experiment completed and saved!")


Training 10% VLE-only leakage model

VLE-ONLY ABLATION RESULTS:
cutoff: 10
condition: VLE_Only
accuracy: 0.6320275661951397
balanced_accuracy: 0.5010231289118617
macro_f1: 0.5169098250055975

10% VLE-only experiment completed and saved!


In [25]:

from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import pandas as pd
import joblib
import os

cutoff = 10

for condition in ["Leakage_Free", "Combined_Leaky"]:

    train = ablation_splits[cutoff][condition]["train"]
    test = ablation_splits[cutoff][condition]["test"]

    X_train = train[features]
    y_train = train["final_result"].map(target_mapping).astype(int)

    X_test = test[features]
    y_test = test["final_result"].map(target_mapping).astype(int)

    model = clone(original_pipeline)

    assert set(train["id_student"]).isdisjoint(
        set(test["id_student"])
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    result = {
        "cutoff": cutoff,
        "condition": condition,
        "accuracy": float(accuracy_score(y_test, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
        "macro_f1": float(f1_score(y_test, pred, average="macro"))
    }

    pd.DataFrame([result]).to_csv(
        f"{ABLATION_DIR}/metrics_{cutoff}pct_{condition}.csv",
        index=False
    )

    joblib.dump(
        model,
        f"{ABLATION_DIR}/model_{cutoff}pct_{condition}.joblib"
    )

    print(f"\n{condition} completed:")
    print(result)

print("\nAll four 10% ablation conditions are now available.")

# Display the complete four-condition comparison
rows = []

for condition in [
    "Leakage_Free",
    "Assessment_Only",
    "VLE_Only",
    "Combined_Leaky"
]:
    path = f"{ABLATION_DIR}/metrics_10pct_{condition}.csv"
    rows.append(pd.read_csv(path).iloc[0].to_dict())

comparison = pd.DataFrame(rows)

display(comparison)

print("\n10% ablation comparison completed!")



Leakage_Free completed:
{'cutoff': 10, 'condition': 'Leakage_Free', 'accuracy': 0.5359085963003264, 'balanced_accuracy': 0.39896501326547007, 'macro_f1': 0.4088475948501647}

Combined_Leaky completed:
{'cutoff': 10, 'condition': 'Combined_Leaky', 'accuracy': 0.7054769677185346, 'balanced_accuracy': 0.6263949423358239, 'macro_f1': 0.6425482684521624}

All four 10% ablation conditions are now available.


,cutoff,condition,accuracy,balanced_accuracy,macro_f1
0,10,Leakage_Free,0.535909,0.398965,0.408848
1,10,Assessment_Only,0.698767,0.619376,0.635544
2,10,VLE_Only,0.632028,0.501023,0.516910
3,10,Combined_Leaky,0.705477,0.626395,0.642548



10% ablation comparison completed!


In [26]:

from sklearn.base import clone
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
import pandas as pd
import joblib
import os

conditions = [
    "Leakage_Free",
    "Assessment_Only",
    "VLE_Only",
    "Combined_Leaky"
]

for cutoff in [20, 30, 40]:
    for condition in conditions:
        print(f"\nTraining {cutoff}% - {condition}")

        train = ablation_splits[cutoff][condition]["train"]
        test = ablation_splits[cutoff][condition]["test"]

        X_train = train[features]
        y_train = train["final_result"].map(target_mapping).astype(int)

        X_test = test[features]
        y_test = test["final_result"].map(target_mapping).astype(int)

        assert set(train["id_student"]).isdisjoint(
            set(test["id_student"])
        )

        model = clone(original_pipeline)
        model.fit(X_train, y_train)

        pred = model.predict(X_test)

        result = {
            "cutoff": cutoff,
            "condition": condition,
            "accuracy": float(accuracy_score(y_test, pred)),
            "balanced_accuracy": float(
                balanced_accuracy_score(y_test, pred)
            ),
            "macro_f1": float(
                f1_score(y_test, pred, average="macro")
            )
        }

        suffix = f"{cutoff}pct_{condition}"

        pd.DataFrame([result]).to_csv(
            f"{ABLATION_DIR}/metrics_{suffix}.csv",
            index=False
        )

        joblib.dump(
            model,
            f"{ABLATION_DIR}/model_{suffix}.joblib"
        )

        print(
            f"Completed: Accuracy={result['accuracy']:.4f}, "
            f"Macro-F1={result['macro_f1']:.4f}"
        )

print("\nAll additional ablation models completed!")

# Consolidate all 16 experiments, including the 10% results
all_results = []

for cutoff in [10, 20, 30, 40]:
    for condition in conditions:
        path = (
            f"{ABLATION_DIR}/"
            f"metrics_{cutoff}pct_{condition}.csv"
        )
        all_results.append(
            pd.read_csv(path).iloc[0].to_dict()
        )

ablation_summary = pd.DataFrame(all_results)

ablation_summary.to_csv(
    f"{ABLATION_DIR}/ablation_all_cutoffs.csv",
    index=False
)

display(ablation_summary)

print("\nComplete ablation results saved successfully!")



Training 20% - Leakage_Free
Completed: Accuracy=0.5781, Macro-F1=0.4480

Training 20% - Assessment_Only
Completed: Accuracy=0.7096, Macro-F1=0.6360

Training 20% - VLE_Only
Completed: Accuracy=0.6467, Macro-F1=0.5301

Training 20% - Combined_Leaky
Completed: Accuracy=0.7142, Macro-F1=0.6410

Training 30% - Leakage_Free
Completed: Accuracy=0.6172, Macro-F1=0.4624

Training 30% - Assessment_Only
Completed: Accuracy=0.7213, Macro-F1=0.6295

Training 30% - VLE_Only
Completed: Accuracy=0.6621, Macro-F1=0.5234

Training 30% - Combined_Leaky
Completed: Accuracy=0.7217, Macro-F1=0.6303

Training 40% - Leakage_Free
Completed: Accuracy=0.6504, Macro-F1=0.4770

Training 40% - Assessment_Only
Completed: Accuracy=0.7321, Macro-F1=0.6178

Training 40% - VLE_Only
Completed: Accuracy=0.6822, Macro-F1=0.5171

Training 40% - Combined_Leaky
Completed: Accuracy=0.7301, Macro-F1=0.6102

All additional ablation models completed!


,cutoff,condition,accuracy,balanced_accuracy,macro_f1
0,10,Leakage_Free,0.535909,0.398965,0.408848
1,10,Assessment_Only,0.698767,0.619376,0.635544
2,10,VLE_Only,0.632028,0.501023,0.516910
3,10,Combined_Leaky,0.705477,0.626395,0.642548
4,20,Leakage_Free,0.578087,0.433449,0.448012
5,20,Assessment_Only,0.709641,0.616191,0.636020
6,20,VLE_Only,0.646683,0.508325,0.530127
7,20,Combined_Leaky,0.714151,0.620861,0.640972
8,30,Leakage_Free,0.617157,0.452400,0.462388
9,30,Assessment_Only,0.721340,0.606112,0.629471



Complete ablation results saved successfully!


In [27]:

import pandas as pd
import os

path = f"{ABLATION_DIR}/ablation_all_cutoffs.csv"
df = pd.read_csv(path)

accuracy = df.pivot(
    index="cutoff",
    columns="condition",
    values="accuracy"
)

result = pd.DataFrame({
    "Cutoff (%)": accuracy.index,
    "Assessment-only inflation (pp)": (
        accuracy["Assessment_Only"] -
        accuracy["Leakage_Free"]
    ).values * 100,
    "VLE-only inflation (pp)": (
        accuracy["VLE_Only"] -
        accuracy["Leakage_Free"]
    ).values * 100,
    "Combined inflation (pp)": (
        accuracy["Combined_Leaky"] -
        accuracy["Leakage_Free"]
    ).values * 100
})

result = result.round(2)

print("FEATURE-FAMILY LEAKAGE CONTRIBUTIONS")
print(result.to_string(index=False))

output = os.path.join(
    ABLATION_DIR,
    "Table_Feature_Family_Leakage_Contributions.csv"
)

result.to_csv(output, index=False)

print("\nPublication-ready table saved successfully!")


FEATURE-FAMILY LEAKAGE CONTRIBUTIONS
 Cutoff (%)  Assessment-only inflation (pp)  VLE-only inflation (pp)  Combined inflation (pp)
         10                           16.29                     9.61                    16.96
         20                           13.16                     6.86                    13.61
         30                           10.42                     4.49                    10.46
         40                            8.17                     3.18                     7.98

Publication-ready table saved successfully!


In [28]:

import joblib
import pandas as pd
from sklearn.metrics import accuracy_score

rows = []

for cutoff in [10, 20, 30, 40]:
    for condition, ablation_condition in [
        ("lf", "Leakage_Free"),
        ("leaky", "Combined_Leaky")
    ]:
        path = f"{BASE}/model_artifacts/xgb_{condition}_{cutoff}.joblib"
        model = joblib.load(path)

        test = ablation_splits[cutoff][ablation_condition]["test"]

        X_test = test[features]
        y_test = test["final_result"].map(target_mapping).astype(int)

        predictions = model.predict(X_test)

        rows.append({
            "Cutoff": cutoff,
            "Condition": condition,
            "Saved model accuracy": accuracy_score(
                y_test, predictions
            )
        })

verification = pd.DataFrame(rows)

print("ORIGINAL SAVED MODEL ACCURACY VERIFICATION")
display(verification)

verification.to_csv(
    f"{ABLATION_DIR}/original_saved_model_verification.csv",
    index=False
)

print("\nOriginal model verification completed!")


ORIGINAL SAVED MODEL ACCURACY VERIFICATION


,Cutoff,Condition,Saved model accuracy
0,10,lf,0.535909
1,10,leaky,0.705477
2,20,lf,0.578087
3,20,leaky,0.714151
4,30,lf,0.617157
5,30,leaky,0.721727
6,40,lf,0.650358
7,40,leaky,0.730111



Original model verification completed!


In [29]:

import pandas as pd
import os

path = (
    f"{BASE}/results/"
    "xgboost_grouped_tuning_10pct_LF.csv"
)

tuning = pd.read_csv(path)

print("TUNING RECORD:")
print("Total candidates:", len(tuning))

# Identify the highest-scoring configuration
best = tuning.loc[tuning["mean_test_score"].idxmax()]

print("\nBEST CROSS-VALIDATION SCORE:")
print(best["mean_test_score"])

print("\nBEST HYPERPARAMETERS:")
for col in tuning.columns:
    if col.startswith("param_model__"):
        print(f"{col}: {best[col]}")

print("\nVerification completed!")


TUNING RECORD:
Total candidates: 15

BEST CROSS-VALIDATION SCORE:
0.40616550488376

BEST HYPERPARAMETERS:
param_model__subsample: 0.7
param_model__reg_lambda: 10
param_model__reg_alpha: 0.01
param_model__n_estimators: 200
param_model__min_child_weight: 7
param_model__max_depth: 6
param_model__learning_rate: 0.08
param_model__gamma: 0.1
param_model__colsample_bytree: 1.0

Verification completed!


In [30]:

import os
import pandas as pd

base = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

matches = []

for root, dirs, files in os.walk(base):
    for filename in files:
        name = filename.lower()
        if filename.endswith(".csv") and (
            "independent_tuning" in name
            or "tuning_sensitivity" in name
        ):
            matches.append(os.path.join(root, filename))

print("INDEPENDENT TUNING RESULT FILES")

for path in sorted(matches):
    print("\nFILE:", path)
    try:
        df = pd.read_csv(path)
        print("Columns:", df.columns.tolist())
        print(df.head(12).to_string(index=False))
    except Exception as e:
        print("Could not read:", str(e))


INDEPENDENT TUNING RESULT FILES

FILE: /content/drive/MyDrive/AI_Research_Paper_1/results/reviewer_sensitivity/independent_tuning_accuracy_comparison.csv
Columns: ['cutoff', 'LF', 'Leaky', 'Inflation_pp']
 cutoff       LF    Leaky  Inflation_pp
     10 0.531375 0.710011     17.863620
     20 0.579402 0.710205     13.080248
     30 0.609799 0.718435     10.863672
     40 0.642204 0.732697      9.049324

FILE: /content/drive/MyDrive/AI_Research_Paper_1/results/reviewer_sensitivity/independent_tuning_all_cutoffs.csv
Columns: ['cutoff', 'condition', 'accuracy', 'balanced_accuracy', 'macro_f1', 'best_cv_macro_f1']
 cutoff condition  accuracy  balanced_accuracy  macro_f1  best_cv_macro_f1
     10        LF  0.531375           0.407146  0.419765          0.407706
     10     Leaky  0.710011           0.630095  0.647185          0.634646
     20        LF  0.579402           0.442776  0.458622          0.435511
     20     Leaky  0.710205           0.613072  0.634483          0.626477
     30 

In [31]:

import os

folder = os.path.join(os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1"), "results", "reviewer_sensitivity")

files = [
    "independent_tuning_all_cutoffs.csv",
    "ablation_all_cutoffs.csv"
]

for filename in files:
    path = os.path.join(folder, filename)
    if os.path.exists(path):
        print("FOUND:", filename)
        print("Size:", os.path.getsize(path), "bytes")
    else:
        print("MISSING:", filename)


FOUND: independent_tuning_all_cutoffs.csv
Size: 733 bytes
MISSING: ablation_all_cutoffs.csv


In [32]:

import os

base = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

print("SEARCHING FOR ABLATION RESULT FILES\n")

found = []

for root, dirs, files in os.walk(base):
    for filename in files:
        if filename.lower().endswith(".csv") and any(
            word in filename.lower()
            for word in ["ablation", "feature_family", "sensitivity"]
        ):
            path = os.path.join(root, filename)
            found.append(path)
            print(path)

print("\nTotal matching CSV files:", len(found))


SEARCHING FOR ABLATION RESULT FILES

/content/drive/MyDrive/AI_Research_Paper_1/results/reviewer_sensitivity/ablation/ablation_all_cutoffs.csv
/content/drive/MyDrive/AI_Research_Paper_1/results/reviewer_sensitivity/ablation/Table_Feature_Family_Leakage_Contributions.csv

Total matching CSV files: 2


In [33]:

import os
import pandas as pd

base = os.path.join(os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1"), "results", "reviewer_sensitivity")

files = {
    "Independent tuning": os.path.join(
        base, "independent_tuning_all_cutoffs.csv"
    ),
    "Feature-family ablation": os.path.join(
        base, "ablation", "ablation_all_cutoffs.csv"
    )
}

for name, path in files.items():
    df = pd.read_csv(path)

    print(f"\n=== {name} ===")
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print(df.to_string(index=False))
    print("\nMissing values:", df.isna().sum().sum())



=== Independent tuning ===
Shape: (8, 6)
Columns: ['cutoff', 'condition', 'accuracy', 'balanced_accuracy', 'macro_f1', 'best_cv_macro_f1']
 cutoff condition  accuracy  balanced_accuracy  macro_f1  best_cv_macro_f1
     10        LF  0.531375           0.407146  0.419765          0.407706
     10     Leaky  0.710011           0.630095  0.647185          0.634646
     20        LF  0.579402           0.442776  0.458622          0.435511
     20     Leaky  0.710205           0.613072  0.634483          0.626477
     30        LF  0.609799           0.460307  0.471198          0.460568
     30     Leaky  0.718435           0.601352  0.624622          0.627056
     40        LF  0.642204           0.478792  0.490494          0.476955
     40     Leaky  0.732697           0.586893  0.612436          0.614423

Missing values: 0

=== Feature-family ablation ===
Shape: (16, 5)
Columns: ['cutoff', 'condition', 'accuracy', 'balanced_accuracy', 'macro_f1']
 cutoff       condition  accuracy  balan

In [34]:

from google.colab import files
import os
import zipfile

base = os.path.join(os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1"), "results", "reviewer_sensitivity")

selected_files = [
    os.path.join(base, "independent_tuning_all_cutoffs.csv"),
    os.path.join(base, "ablation", "ablation_all_cutoffs.csv"),
    os.path.join(
        base, "ablation",
        "Table_Feature_Family_Leakage_Contributions.csv"
    )
]

zip_path = "/content/reviewer_sensitivity_results.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for path in selected_files:
        if not os.path.isfile(path):
            raise FileNotFoundError(path)
        z.write(path, arcname=os.path.basename(path))

print("ZIP created successfully!")
print("Files included:")
for path in selected_files:
    print("-", os.path.basename(path))

files.download(zip_path)


ZIP created successfully!
Files included:
- independent_tuning_all_cutoffs.csv
- ablation_all_cutoffs.csv
- Table_Feature_Family_Leakage_Contributions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:

import json
from pathlib import Path

notebook_path = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

print("Project folder:", notebook_path)
print("\nPath portability will be checked separately.")
print("No notebook files have been modified.")


Project folder: /content/drive/MyDrive/AI_Research_Paper_1

Path portability will be checked separately.
No notebook files have been modified.


In [36]:

import json
from pathlib import Path

# Locate the notebook in Google Drive
matches = list(
    Path("/content/drive/MyDrive").rglob(
        "08_Reviewer_Sensitivity_Analyses.ipynb"
    )
)

if not matches:
    print("Notebook file not found in Google Drive.")
else:
    notebook_file = matches[0]
    print("Notebook:", notebook_file)

    with open(notebook_file, "r", encoding="utf-8") as f:
        notebook = json.load(f)

    print("\nHARD-CODED PATHS IN CODE CELLS")

    count = 0

    for cell_number, cell in enumerate(
        notebook.get("cells", []), start=1
    ):
        if cell.get("cell_type") != "code":
            continue

        source = "".join(cell.get("source", []))

        for line in source.splitlines():
            if "/content/drive/MyDrive/" in line:
                count += 1
                print(f"Cell {cell_number}: {line.strip()}")

    print("\nTotal executable path references:", count)
    print("No files were modified.")


Notebook: /content/drive/MyDrive/Colab Notebooks/08_Reviewer_Sensitivity_Analyses.ipynb

HARD-CODED PATHS IN CODE CELLS
Cell 1: BASE_PATH = "/content/drive/MyDrive/AI_Research_Paper_1"
Cell 2: BASE = "/content/drive/MyDrive/AI_Research_Paper_1"
Cell 3: BASE = "/content/drive/MyDrive/AI_Research_Paper_1"
Cell 4: BASE = "/content/drive/MyDrive/AI_Research_Paper_1"
Cell 5: BASE = "/content/drive/MyDrive/AI_Research_Paper_1"
Cell 6: BASE = "/content/drive/MyDrive/AI_Research_Paper_1"
Cell 7: BASE = "/content/drive/MyDrive/AI_Research_Paper_1"
Cell 8: BASE = "/content/drive/MyDrive/AI_Research_Paper_1"
Cell 19: BASE = "/content/drive/MyDrive/AI_Research_Paper_1"
Cell 20: BASE = "/content/drive/MyDrive/AI_Research_Paper_1"
Cell 30: base = "/content/drive/MyDrive/AI_Research_Paper_1"
Cell 31: folder = "/content/drive/MyDrive/AI_Research_Paper_1/results/reviewer_sensitivity"
Cell 32: base = "/content/drive/MyDrive/AI_Research_Paper_1"
Cell 33: base = "/content/drive/MyDrive/AI_Research_Paper_1